# 🎯 Module 10: Transformers Basics

> **Stage:** Deep Learning  
> **Level:** Intermediate → Job-Ready  
> **Module:** Transformers Basics  
> **Framework:** PyTorch  
> **Date:** 10 June 2026  
> **Previous:** 09 — LSTM & GRU  
> **Next:** 11 — Transfer Learning

---

Transformers are the architecture behind **modern LLMs, vision models, speech systems, and multimodal AI**.
In this notebook we'll build them from the ground up — starting from the attention equation and ending with a working
Transformer classifier trained in PyTorch.

We will cover:

- Why Transformers replaced RNNs
- **Self-attention**: Query, Key, Value
- **Scaled dot-product attention** (the core equation)
- Why we divide by √dₖ
- **Multi-head attention**
- **Positional information**
- Residual connections & LayerNorm
- Feed-forward networks
- Encoder vs Decoder architectures
- Causal masking
- `nn.TransformerEncoder` in PyTorch
- Building & training a small Transformer classifier

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain why Transformers beat RNNs on long-range sequence tasks
- Write the **scaled dot-product attention** equation from memory
- Explain Query / Key / Value intuition
- Compute attention scores by hand on a tiny example
- Explain the role of multi-head attention
- Explain why positional information is needed
- Describe residual connections and LayerNorm
- Distinguish encoder-only, decoder-only, encoder-decoder
- Apply causal masking for autoregressive generation
- Use `nn.TransformerEncoderLayer` and `nn.TransformerEncoder`
- Build, train and evaluate a small Transformer classifier

---
## ⚙️ 0. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

---
## 🧠 1. What Is a Transformer?

A **Transformer** is a neural network architecture designed for **sequence data** using **attention** instead of recurrence.

Used for:
- NLP (translation, generation, classification)
- Computer vision (ViT)
- Speech recognition
- Multimodal AI
- **Large Language Models**

### The key idea

> Instead of processing tokens strictly one after another, **attention lets each token directly interact with every other relevant token**.

### RNN vs Transformer

```text
RNN:         x₁ → x₂ → x₃ → x₄ → x₅
Transformer: x₁ ─────┐
             x₂ ─────┤
             x₃ ─────┼──→ Attention
             x₄ ─────┤
             x₅ ─────┘
```

Benefits:
- Better **long-range dependencies**
- **Highly parallelizable** during training

---
## 💬 2. Motivating Example

Consider:

```text
The animal didn't cross the road because it was tired.
```

What does `it` refer to? — The **animal**.

Attention allows the representation of `it` to directly consider other words and learn these relationships.

```text
Token
 ↓
Compare with other tokens
 ↓
Determine relevance
 ↓
Combine useful information
```

---
## 🔍 3. Self-Attention & Q / K / V

**Self-attention** means a sequence attends to itself.

For every token, we create three learned projections:

$$Q = X W_Q, \quad K = X W_K, \quad V = X W_V$$

### Intuition

| | Meaning |
| --- | --- |
| **Query** | What information am I looking for? |
| **Key** | What information does this token represent? |
| **Value** | What information should actually be returned? |

```text
Query
  ↓
compare with Keys
  ↓
attention scores
  ↓
weighted Values
```

Think of it like a **soft search system**: queries search through keys, and results are a weighted combination of values.

---
## ⭐ 4. Scaled Dot-Product Attention

The central equation of the Transformer:

$$\boxed{\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V}$$

### Step by step

```text
QKᵀ
 ↓
Similarity scores
 ↓
Scale by √dₖ
 ↓
Softmax
 ↓
Attention weights
 ↓
Multiply by V
 ↓
Output
```

### Why divide by √dₖ?

As the key/query dimension grows, dot products get large, which pushes Softmax into a very sharp distribution (near one-hot) — killing gradients.
Dividing by $\sqrt{d_k}$ keeps the values in a healthy range for stable optimization.

In [ ]:
# Implement scaled dot-product attention from scratch
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: [..., T_q, d_k]
    K: [..., T_k, d_k]
    V: [..., T_k, d_v]
    mask: optional boolean tensor, True = block
    """
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)     # [..., T_q, T_k]
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

# Tiny example: T=4 tokens, d_k=8
T, d_k, d_v = 4, 8, 8
Q = torch.randn(T, d_k)
K = torch.randn(T, d_k)
V = torch.randn(T, d_v)

out, weights = scaled_dot_product_attention(Q, K, V)
print('Output shape :', out.shape)
print('Weights shape:', weights.shape)
print('Row sums (should be 1.0):', weights.sum(dim=-1))

In [ ]:
# Attention weights interpretation on a tiny example
scores = torch.tensor([2.0, 1.0, 0.0])
w = F.softmax(scores, dim=0)
print('Raw scores:', scores.tolist())
print('Softmax   :', [round(x, 3) for x in w.tolist()])
print()
for i, wi in enumerate(w):
    print(f'  {wi.item()*100:5.1f}% attention → token {i+1}')

In [ ]:
# Visualize an attention matrix
torch.manual_seed(0)
T = 8
Q = torch.randn(T, 16)
K = torch.randn(T, 16)
V = torch.randn(T, 16)
_, W = scaled_dot_product_attention(Q, K, V)

plt.figure(figsize=(5, 4))
plt.imshow(W.detach().numpy(), cmap='viridis')
plt.colorbar(label='attention weight')
plt.title('Attention weights (query × key)')
plt.xlabel('Key position'); plt.ylabel('Query position')
plt.tight_layout(); plt.show()

---
## 🎩 5. Multi-Head Attention

Instead of a single attention operation, Transformers use **multiple attention heads** in parallel:

```text
Input
  ↓
┌──────┬──────┬──────┐
Head 1 Head 2 Head 3 ...
└──────┴──────┴──────┘
       ↓
    Concatenate
       ↓
   Linear layer
```

### Why multiple heads?

Each head can attend to different types of relationships:

```text
Head 1 → local relationships
Head 2 → grammatical relationships
Head 3 → semantic relationships
```

⚠️ These are intuitions — the model learns its own head specializations.

In [ ]:
# Multi-head attention using PyTorch's built-in module
d_model, n_heads = 32, 4
mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, batch_first=True)

x = torch.randn(2, 6, d_model)      # [B, T, d_model]
out, attn_weights = mha(x, x, x)    # self-attention

print('Input  :', x.shape)
print('Output :', out.shape)
print('Weights:', attn_weights.shape, '  → [B, n_heads, T, T]')

---
## 📍 6. Positional Information

Attention itself does **not** know sequence order!

```text
dog bites man
man bites dog
```

Same tokens, different meaning. So we must **inject position**.

### Sinusoidal positional encoding (original Transformer)

$$PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

Modern Transformers use alternatives (learned, RoPE, ALiBi), but the **requirement** is universal: the model must know where each token sits.

In [ ]:
# Visualize sinusoidal positional encoding
def positional_encoding(T, d_model):
    pe = torch.zeros(T, d_model)
    pos = torch.arange(0, T).unsqueeze(1).float()          # [T, 1]
    div = torch.exp(torch.arange(0, d_model, 2).float() *
                    (-math.log(10000.0) / d_model))         # [d_model/2]
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

pe = positional_encoding(T=50, d_model=64)
plt.figure(figsize=(8, 4))
plt.imshow(pe.numpy(), aspect='auto', cmap='RdBu')
plt.colorbar()
plt.title('Sinusoidal positional encoding (T=50, d=64)')
plt.xlabel('dimension'); plt.ylabel('position')
plt.tight_layout(); plt.show()

---
## 🧱 7. Transformer Block

```text
Input
  ↓
Multi-Head Self-Attention
  ↓
Add & LayerNorm        ← residual
  ↓
Feed-Forward Network
  ↓
Add & LayerNorm        ← residual
  ↓
Output
```

### Residual connections

$$y = x + F(x)$$

```text
x ───────────────┐
 ↓               +
Layer ───────────┘
```

Residuals help gradients and information flow through deep stacks.

### LayerNorm

```python
nn.LayerNorm(d_model)
```

Normalizes features **within each example** — ideal for sequences.

### Feed-Forward Network

$$\text{FFN}(x) = W_2 \sigma(W_1 x + b_1) + b_2$$

Applied **independently** to each token position.

In [ ]:
# Build a single Transformer encoder layer and run a forward pass
layer = nn.TransformerEncoderLayer(
    d_model=32,
    nhead=4,
    dim_feedforward=64,
    dropout=0.1,
    batch_first=True,
)

x = torch.randn(2, 6, 32)      # [B, T, d_model]
y = layer(x)
print('Input :', x.shape)
print('Output:', y.shape)

---
## 🏗️ 8. Encoder vs Decoder

The original Transformer has both:

```text
Input
 ↓
Encoder
 ↓
Representations
 ↓
Decoder
 ↓
Output
```

### Modern configurations

| Type | Main idea | Example |
| --- | --- | --- |
| **Encoder-only** | Understand input | BERT, classification |
| **Decoder-only** | Generate sequence | GPT, LLMs |
| **Encoder-decoder** | Transform input → output | T5, translation |

Large language models commonly use **decoder-only** architectures for autoregressive generation.

---
## 🙈 9. Causal Masking

For autoregressive generation, a token must **not** see future tokens.

Example: `I → can → build → AI`

When predicting `build`, the model should not peek at `AI`.

A **causal mask** enforces this:

```text
✓ = can attend
✗ = cannot attend

Token 1 → ✓ ✗ ✗ ✗
Token 2 → ✓ ✓ ✗ ✗
Token 3 → ✓ ✓ ✓ ✗
Token 4 → ✓ ✓ ✓ ✓
```

This is also called **causal self-attention**.

In [ ]:
# Build a causal mask and apply it
T = 5
causal_mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
print('Causal mask (True = blocked):')
print(causal_mask.int())

# Attention with mask
torch.manual_seed(0)
Q = torch.randn(T, 8); K = torch.randn(T, 8); V = torch.randn(T, 8)
out, W = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
print('\nAttention weights (masked):')
print(W.detach().round(decimals=2))
print('\nNote: upper triangle is zero → tokens cannot attend to the future.')

---
## ⏱️ 10. Complexity — The O(n²) Problem

Standard self-attention compares **every token** with **every other token**:

$$\text{Complexity} = O(n^2)$$

where `n` is sequence length.

If sequence length doubles, pairwise attention work grows **4×**:

```text
n  → 2n
n² → 4n²
```

That's why **long-context efficiency** is a major research area (sparse attention, sliding windows, linear attention, FlashAttention, …).

---
## 🛠️ 11. Building a Small Transformer Classifier

Pipeline:

```text
Input tokens
     ↓
Embedding
     ↓
Positional information
     ↓
Transformer Encoder
     ↓
Pooling
     ↓
Linear
     ↓
Classification
```

We'll build a synthetic sequence classification task: sequences whose **mean signal is > 0** are class 1.

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size=None, d_model=64, nhead=4,
                 num_layers=2, num_classes=2, seq_len=20, input_dim=1):
        """
        Two modes:
          - input_dim given: continuous features [B, T, input_dim] → Linear projection
          - vocab_size given: integer tokens [B, T]       → Embedding
        """
        super().__init__()
        if vocab_size is not None:
            self.embed = nn.Embedding(vocab_size, d_model)
        else:
            self.embed = nn.Linear(input_dim, d_model)

        self.pos = nn.Parameter(torch.zeros(1, seq_len, d_model))
        nn.init.trunc_normal_(self.pos, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=0.1, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.embed(x) + self.pos[:, :x.size(1), :]
        x = self.encoder(x)
        x = self.norm(x.mean(dim=1))       # mean-pool over time
        return self.head(x)

# Shape check
m = TransformerClassifier(d_model=64, nhead=4, num_layers=2, input_dim=8, seq_len=20, num_classes=2)
x = torch.randn(4, 20, 8)
print('Input :', x.shape)
print('Output:', m(x).shape)
print('Params:', sum(p.numel() for p in m.parameters()), 'total')

In [ ]:
# Synthetic sequence classification data
torch.manual_seed(42)
N_TRAIN, N_VAL = 2000, 500
T, F = 20, 8

def make_seq_data(n):
    X = torch.randn(n, T, F)
    signal = X.mean(dim=(1, 2))          # mean over time AND features
    y = (signal > 0).long()
    return X, y

X_train, y_train = make_seq_data(N_TRAIN)
X_val,   y_val   = make_seq_data(N_VAL)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=128, shuffle=False)

print('Train:', X_train.shape, 'Val:', X_val.shape)

In [ ]:
# Train the Transformer classifier
def train_one_epoch(model, loader, optimizer, criterion, clip=1.0):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        if clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        correct    += (logits.argmax(1) == yb).sum().item()
        total      += xb.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)
        correct    += (logits.argmax(1) == yb).sum().item()
        total      += xb.size(0)
    return total_loss / total, correct / total

torch.manual_seed(42)
model = TransformerClassifier(d_model=64, nhead=4, num_layers=2,
                              input_dim=F, seq_len=T, num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 10
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    va_loss, va_acc = evaluate(model, val_loader, criterion)
    history['train_loss'].append(tr_loss); history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc);   history['val_acc'].append(va_acc)
    print(f'Epoch {epoch:2d}/{EPOCHS} | '
          f'train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f} | '
          f'val_loss={va_loss:.4f}   val_acc={va_acc:.4f}')

In [ ]:
# Learning curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'],   label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history['train_acc'], label='train')
axes[1].plot(history['val_acc'],   label='val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Inspect the attention pattern of the trained model
model.eval()
xb, yb = next(iter(val_loader))
xb = xb.to(device)

with torch.no_grad():
    h = model.embed(xb[:1]) + model.pos[:, :xb.size(1), :]
    # Get attention weights from the first encoder layer
    attn_out, attn_w = model.encoder.layers[0].self_attn(h, h, h, need_weights=True)

print('Attention weights shape:', attn_w.shape)   # [B, n_heads, T, T]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for i, ax in enumerate(axes):
    ax.imshow(attn_w[0, i].cpu().numpy(), cmap='viridis')
    ax.set_title(f'Head {i+1}')
    ax.set_xlabel('Key'); ax.set_ylabel('Query')
plt.suptitle('Attention heads from a trained Transformer layer')
plt.tight_layout(); plt.show()

---
## ⚖️ 12. Transformer vs RNN

| Feature | RNN | Transformer |
| --- | --- | --- |
| Recurrence | ✅ Yes | ❌ No |
| Hidden state | ✅ Yes | ❌ No recurrent state |
| Self-attention | ❌ No | ✅ Yes |
| Parallel training | ❌ Limited | ✅ Strong |
| Long dependencies | Difficult | Better |
| Main mechanism | Recurrence | Attention |

Transformers still pay a cost for long sequences: standard self-attention is **O(n²)**.

---
## 🚨 13. Common Transformer Mistakes

### ❌ 1. Forgetting positional information
Without it, the model is **permutation-invariant** — order is invisible.

### ❌ 2. Wrong shape with `batch_first`
Set `batch_first=True` if you feed `[B, T, D]`, else PyTorch expects `[T, B, D]`.

### ❌ 3. Missing causal mask for generation
Without a causal mask, the decoder can peek at the future → cheating during training.

### ❌ 4. `d_model` not divisible by `nhead`
`nn.MultiheadAttention` requires `d_model % nhead == 0`.

### ❌ 5. Dropping residual / LayerNorm
Deep Transformers won't train without them.

### ❌ 6. Ignoring O(n²) cost
Attention memory/time explodes with sequence length — plan for it.

---
## 🏋️ 14. Hands-On Exercises

### Exercise 1 — Attention by Hand
Given `Q`, `K`, `V` as 3×2 matrices, compute `softmax(QKᵀ / √dₖ) V` by hand, then verify with the `scaled_dot_product_attention` function above.

### Exercise 2 — Sweep `d_model`, `nhead`, `num_layers`
Train with:
- `d_model ∈ {32, 64, 128}`
- `nhead ∈ {2, 4, 8}` (must divide `d_model`)
- `num_layers ∈ {1, 2, 3}`

Compare val accuracy, params, training time.

### Exercise 3 — Learned vs Sinusoidal Positions
Replace the learned `self.pos` parameter with the sinusoidal encoding function defined above. Any difference?

### Exercise 4 — Decoder-only LM
Use `nn.TransformerEncoderLayer` with a causal mask (`is_causal=True`) on a synthetic next-token task.

### Exercise 5 — Max-Pool vs Mean-Pool
Change the classifier head from `x.mean(dim=1)` to `x.max(dim=1).values`. Compare.

### Exercise 6 — Token Embedding Mode
Use the `vocab_size` branch (integer tokens) and train on a small text classification task.

### Exercise 7 — Attention Visualization
During training, snapshot attention maps for a specific head across epochs. Do patterns stabilize?

---
## 🎤 15. Interview Questions

1. **What is self-attention?**  
   A mechanism where each token calculates how strongly it should use information from other tokens in the same sequence.

2. **What are Q, K, and V?**  
   Learned Query, Key, and Value projections used to compute and apply attention.

3. **Why use multiple attention heads?**  
   To let the model learn different relationships in separate attention subspaces.

4. **Why divide by √dₖ?**  
   To keep dot-product magnitudes in a stable range before Softmax.

5. **Why is positional information needed?**  
   Attention alone does not encode sequence order.

6. **Why residual connections?**  
   They improve information and gradient flow in deep stacks.

7. **Why LayerNorm?**  
   To stabilize representations during training, especially across sequence positions.

8. **Why do Transformers beat RNNs on large-scale sequence tasks?**  
   Direct token-to-token interaction + highly parallelizable training.

9. **What is causal masking?**  
   Masking future positions so autoregressive models cannot peek ahead.

10. **What's the main limitation of self-attention?**  
    O(n²) time/memory with sequence length.

---
## ✅ 16. Knowledge Check

- [ ] Explain why Transformers replaced RNNs
- [ ] Write the scaled dot-product attention equation
- [ ] Explain Query / Key / Value intuition
- [ ] Explain why we divide by √dₖ
- [ ] Explain multi-head attention
- [ ] Explain why positional information is needed
- [ ] Draw a Transformer block
- [ ] Explain residual connections and LayerNorm
- [ ] Explain encoder-only vs decoder-only vs encoder-decoder
- [ ] Explain causal masking
- [ ] Compute attention by hand for a small example
- [ ] Build and train a Transformer classifier in PyTorch
- [ ] Explain the O(n²) complexity

---
## 🏁 17. Key Takeaways

```text
Transformer
│
├── Self-Attention
│   ├── Query
│   ├── Key
│   └── Value
├── Multi-Head Attention
├── Positional Information
├── Feed-Forward Network
├── Residual Connections
├── Layer Normalization
└── Encoder / Decoder architectures
```

### Core equation

$$\boxed{\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V}$$

### The historical progression

```text
RNN
 ↓
LSTM / GRU
 ↓
Attention
 ↓
Transformer
 ↓
Modern LLMs
```

### 🧭 Golden Rule

> **Attention lets each token dynamically gather information from every other token — but without position and masking, it's blind to order and to the future.**